# 0. Probe: does this pipeline reproduce a reference run?

Nothing is evaluated here and no results are written. This notebook answers two
questions before any long run is started.

**Is our code faithful?** An earlier pass of the same checkpoint with the
upstream evaluator left a detections table for 50 patches. This notebook runs
our own pipeline over the *same* patches with the *same* checkpoint and
compares point by point. Agreement exercises checkpoint loading, the forward
pass and peak extraction together. Disagreement is found in minutes, before the
full run. The reference table is not part of this repository, so section 5 can
only be re-run by whoever holds it; its printed report is kept below.

**How long will the full run take?** Seconds per patch are measured on this
GPU, after warm-up, and projected to the full 2,607-patch split. Reduced
precision is measured too, so the speed-up can be judged against whether it
moves any detections.

Run order: mount Drive, configure, sync the code, start the session, build the
model, then the two checks.


In [1]:
# --- 0. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# --- 1. Configuration ---
ASSETS  = "/content/drive/MyDrive/OWL_Caribou_Project"          # checkpoints + CAH split (only .verified.json markers are written here)
RESULTS = "/content/drive/MyDrive/owl_caribou_overhead/results" # ours to write (nothing written here)

# The reference run to reproduce: an earlier pass of the same checkpoint with the
# upstream evaluator over 50 patches. The table is not in the repository. owl-d
# is the primary model and the heaviest, so it is the honest thing to probe.
REFERENCE_MODEL = "owl-d"
REFERENCE_TABLE = f"{ASSETS}/results/smoke/{REFERENCE_MODEL}/detections.csv"

# caribou-owl-c shares owl-c's architecture, so timing it separately adds nothing.
TIMING_MODELS = ["owl-c", "owl-t", "owl-d"]
WARMUP, MEASURE, BATCH_SIZE = 4, 20, 8   # warm-up is rounded up to a whole batch

from pathlib import Path
if not (Path(ASSETS) / "data" / "test" / "gt.csv").is_file():
    raise RuntimeError(f"{ASSETS}/data/test/gt.csv not found. Put the assets there as the README's "
                       "'Expected assets' section describes, or point ASSETS at where they are.")


In [ ]:
# === CODE SYNC (auto-generated by `python -m owlcaribou.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m owlcaribou.sync   and reopen this notebook.")

In [4]:
# --- 3. Start the session ---
from owlcaribou.session import start_session

session = start_session(assets=ASSETS, results=RESULTS,
                        models=sorted({REFERENCE_MODEL, *TIMING_MODELS}), require_gpu=True)


session: Colab | device cuda (NVIDIA A100-SXM4-40GB, 39.49 GB)
  torch   2.11.0+cu128 | CUDA 12.8 | Python 3.13.15
  assets  /content/drive/MyDrive/OWL_Caribou_Project
  results /content/drive/MyDrive/owl_caribou_overhead/results
  code    c26c4d3f466e
  split   2607 patches, 12456 points
  weights verified: owl-c, owl-d, owl-t


In [5]:
# --- 4. Build the model and load its checkpoint ---
from owlcaribou import infer
from owlcaribou.data import CahPatches, load_ground_truth, cross_check

patches = CahPatches(session.paths.test_data, expect_images=2607)
truth   = load_ground_truth(session.paths.ground_truth)
print("split:", cross_check(patches, truth))

net  = infer.build_model(REFERENCE_MODEL, session.third_party, device=session.device)
spec = infer.MODEL_SPECS[REFERENCE_MODEL]
print("loaded:", infer.load_checkpoint(net, session.paths.checkpoint(REFERENCE_MODEL),
                                       device=session.device))
print("prediction scale:", spec.prediction_scale)

split: {'patches': 2607, 'with_points': 1852, 'background': 755, 'points': 12456, 'mosaics_with_animals': 26}
loaded: {'checkpoint': 'OWL-D.pth', 'tensors': 672, 'unwrapped': 'model.', 'epoch': 15}
prediction scale: 2


In [6]:
# --- 5. Reproduce the reference detections on the same patches ---
from owlcaribou import archive

reference = archive.load_archived_detections(REFERENCE_TABLE)
position  = {name: i for i, name in enumerate(patches.names)}
missing   = [n for n in reference.images if n not in position]
assert not missing, f"{len(missing)} reference patches are not in the split, e.g. {missing[:3]}"

chosen = [position[n] for n in reference.images]
ours   = list(infer.run_inference(net, patches, spec, device=session.device,
                                  batch_size=BATCH_SIZE, num_workers=2, indices=chosen))

# The reference table stores heatmap indices: the upstream evaluator applied
# down_ratio when it scored, not when it wrote the table. We emit patch pixels,
# so the reference is scaled up to match. The two ranges printed below should agree.
report = archive.compare(ours, reference, archived_scale=spec.prediction_scale)

# The report keys follow the archive module; print them as "reference".
label = {"archived_detections": "reference_detections", "archived_range": "reference_range"}
for key in ("patches_compared", "agreeing", "disagreeing", "our_detections",
            "archived_detections", "our_range", "archived_range", "tolerance", "identical"):
    print(f"  {label.get(key, key):20s} {report[key]}")
if report["count_differences"]:
    print("  count differences (patch, ours, reference):", report["count_differences"])
if report["displaced"]:
    print("  displaced points (patch, worst px):", report["displaced"])
if report["only_ours"] or report["only_archived"]:
    print("  patches on one side only (ours, reference):",
          report["only_ours"][:5], report["only_archived"][:5])

print()
print("REPRODUCED" if report["identical"] else
      "DIVERGED - do not start the full run until this is understood")


  patches_compared     50
  agreeing             50
  disagreeing          0
  our_detections       231
  reference_detections 231
  our_range            (0.0, 510.0)
  reference_range      (0.0, 510.0)
  tolerance            0.5
  identical            True

REPRODUCED


In [7]:
# --- 6. How long will the full split take? ---
import torch

# Read the timed patches once first, so every line below measures compute, not Drive.
for index in range(min(len(patches), WARMUP + BATCH_SIZE + MEASURE)):
    patches[index]

timings = {}
for name in TIMING_MODELS:
    model = net if name == REFERENCE_MODEL else infer.build_model(name, session.third_party,
                                                                device=session.device)
    if name != REFERENCE_MODEL:
        infer.load_checkpoint(model, session.paths.checkpoint(name), device=session.device)
    for precision, dtype in (("fp32", None), ("bf16", torch.bfloat16)):
        result = infer.time_inference(model, patches, infer.MODEL_SPECS[name],
                                      device=session.device, warmup=WARMUP, measure=MEASURE,
                                      batch_size=BATCH_SIZE, num_workers=2,
                                      autocast_dtype=dtype)
        timings[(name, precision)] = result
        print(f"{name:14s} {precision}  {result['seconds_per_patch']:6.3f} s/patch  "
              f"-> {result['projected_full_split_minutes']:6.1f} min for 2,607  "
              f"({int(result['detections'])} detections in {int(result['measured_patches'])} patches)")
    if name != REFERENCE_MODEL:
        del model
        torch.cuda.empty_cache()

total = sum(t["projected_full_split_minutes"] for (n, p), t in timings.items()
            if p == "fp32") + timings[("owl-c", "fp32")]["projected_full_split_minutes"]
print(f"projected fp32 total for all four models: {total:.0f} min")
print("bf16 detection counts differing from fp32 above means reduced precision moves results.")

owl-c          fp32   0.007 s/patch  ->    0.3 min for 2,607  (157 detections in 20 patches)
owl-c          bf16   0.008 s/patch  ->    0.3 min for 2,607  (162 detections in 20 patches)


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


owl-t          fp32   0.009 s/patch  ->    0.4 min for 2,607  (106 detections in 20 patches)
owl-t          bf16   0.008 s/patch  ->    0.4 min for 2,607  (108 detections in 20 patches)
owl-d          fp32   0.117 s/patch  ->    5.1 min for 2,607  (140 detections in 20 patches)
owl-d          bf16   0.024 s/patch  ->    1.0 min for 2,607  (150 detections in 20 patches)
projected fp32 total for all four models: 6 min
bf16 detection counts differing from fp32 above means reduced precision moves results.


## What this tells us

**If section 5 says REPRODUCED**, this pipeline matches the reference run on
the same inputs, and the full evaluation can be started with confidence.

**If it says DIVERGED**, stop. The likely causes, in order of how easy they are
to check: a different `down_ratio` than the reference configuration, reduced
precision left on, or a peak-extraction setting that does not match the
evaluation setting. The report names the patches that disagree and by how much.

Section 6 reads its patches once before timing, so its projection is compute
only. The first pass of the full run is slower, because every patch is read
from Drive once: on this run 16 min for OWL-D against the 5 min projected
here (`01_full_cah` records the measured time). Note the detection counts for bf16 against fp32: if they
differ, the speed-up is not free, because the peak threshold is relative to
each patch's own maximum and so is sensitive to small numeric shifts.

**Next:** `01_full_cah.ipynb`, which runs all four models over the whole split
and writes per-model results, resumable after a disconnect.
